In [ ]:
!pip install -U ibm-watsonx-ai pillow

import os, base64, mimetypes
from pathlib import Path
from getpass import getpass
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference


In [ ]:

# 1) Watsonx credentials
API_KEY = ""
PROJECT_ID = ""

# Region URL:
# Dallas: https://us-south.ml.cloud.ibm.com

URL = "https://us-south.ml.cloud.ibm.com"

credentials = Credentials(
    url=URL,
    api_key=API_KEY
)



In [ ]:
# 2) Vision model
model = ModelInference(
    model_id="meta-llama/llama-3-2-11b-vision-instruct",
    credentials=credentials,
    project_id=PROJECT_ID,
    params={
        "max_tokens": 300,
        "temperature": 0.2
    }
)



In [ ]:
# 3) Convert image to base64
def encode_image(image_path):
    image_path = Path(image_path)
    mime_type, _ = mimetypes.guess_type(image_path)

    if mime_type is None:
        mime_type = "image/jpeg"

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"



In [ ]:
# 4) Ask watsonx to identify and describe animal
def describe_animal(image_path):
    image_data = encode_image(image_path)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": """
Identify the animal or animals in this image.
For each animal, provide:
1. Animal name
2. Short physical description
3. What it appears to be doing
4. Confidence level: High, Medium, or Low

If there is no animal, say: No animal detected.
"""
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data
                    }
                }
            ]
        }
    ]

    response = model.chat(messages=messages)
    return response["choices"][0]["message"]["content"]



In [ ]:
# 5) Analyze all images inside a folder
folder_path = "sample_data" # change this folder name

for image_file in Path(folder_path).glob("*"):
    if image_file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
        print("=" * 60)
        print("Image:", image_file.name)
        print(describe_animal(image_file))

Image: images3.jpeg
1. **Animal Name:** Eurasian Eagle Owl
2. **Physical Description:** Large, grayish-brown plumage with white markings on the face and belly, distinctive ear tufts, and piercing yellow eyes.
3. **Activity:** Perched on a rock, gazing intently into the distance.
4. **Confidence Level:** High

The image clearly depicts a Eurasian Eagle Owl, characterized by its distinctive appearance and behavior. The owl's posture and gaze suggest it is engaged in hunting or surveillance, typical of its nocturnal and carnivorous nature.
Image: images.jpeg
**Animal Description**

* **Animal:** German Shepherd Dog
* **Physical Description:** The dog has a medium-length coat with a black and tan coloration, pointed ears, and a long tongue hanging out of its mouth.
* **Activity:** The dog appears to be standing in a grassy area, possibly waiting for a command or treat.
* **Confidence Level:** High
Image: images2.jpeg
**Animal Identification**

Based on the image, I identify the following a